# Signature Verification — Siamese CNN

The plain CNN couldn't really learn *similarity*. A Siamese network fixes that: the same CNN tower
(shared weights) turns each signature into an embedding vector, and we compare the two embeddings
by Euclidean distance.

We train with **contrastive loss** so genuine pairs end up close together and forged pairs end up
far apart. At test time we just pick a distance threshold.

**Techniques used:** shared-weight towers, Conv2D + BatchNorm + Dropout, Lambda distance layer, contrastive loss, EER threshold

## 1. Imports

In [ ]:
import os, json
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

import tensorflow as tf
from keras.models import Sequential, Model
from keras.layers import Input, Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout, BatchNormalization, Lambda
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import roc_auc_score, roc_curve

## 2. Get the Dataset

In [ ]:
# clone the project repo (run once) — the dataset ships inside it
!git clone https://github.com/goyashek/Signature-forgery-verification.git

DATA_ROOT = 'Signature-forgery-verification/sign_data'
IMG_DIR   = os.path.join(DATA_ROOT, 'train')

IMG_H, IMG_W = 150, 220

## 3. Writer-independent split

Same as notebook 1 — the shipped `test/` folder is a duplicate of `train/`, so we split by writer
id instead and test only on unseen people.

In [ ]:
cols = ['img1', 'img2', 'label']
df = pd.concat([pd.read_csv(os.path.join(DATA_ROOT, 'train_data.csv'), header=None, names=cols),
                pd.read_csv(os.path.join(DATA_ROOT, 'test_data.csv'),  header=None, names=cols)])
df = df.drop_duplicates().reset_index(drop=True)
df['writer'] = df['img1'].str.split('/').str[0].astype(int)

train_df = df[df['writer'] <= 40]
val_df   = df[(df['writer'] >= 41) & (df['writer'] <= 48)]
test_df  = df[df['writer'] >= 49]
print('pairs -> train:', len(train_df), '| val:', len(val_df), '| test:', len(test_df))

## 4. Load image pairs

label: **0 = genuine (similar), 1 = forged (different)**. This time we keep the two signatures as
two separate arrays, since each goes through the tower on its own.

In [ ]:
def sample_balanced(frame, n):
    g = frame[frame.label == 0].sample(min(n//2, sum(frame.label==0)), random_state=42)
    f = frame[frame.label == 1].sample(min(n//2, sum(frame.label==1)), random_state=42)
    return pd.concat([g, f]).sample(frac=1, random_state=42)

def load_pairs(frame):
    X1, X2, y = [], [], []
    for _, row in frame.iterrows():
        a = cv2.imread(os.path.join(IMG_DIR, row['img1']), cv2.IMREAD_GRAYSCALE)
        b = cv2.imread(os.path.join(IMG_DIR, row['img2']), cv2.IMREAD_GRAYSCALE)
        a = cv2.resize(a, (IMG_W, IMG_H))
        b = cv2.resize(b, (IMG_W, IMG_H))
        X1.append(a); X2.append(b); y.append(row['label'])
    X1 = np.array(X1)[..., None] / 255.0
    X2 = np.array(X2)[..., None] / 255.0
    return X1, X2, np.array(y)

X1_tr, X2_tr, y_tr = load_pairs(sample_balanced(train_df, 8000))
X1_va, X2_va, y_va = load_pairs(sample_balanced(val_df,   2000))
X1_te, X2_te, y_te = load_pairs(sample_balanced(test_df,  3000))
print('train:', X1_tr.shape, '| test:', X1_te.shape)

## 5. The shared tower

One small CNN that maps a signature to a 128-d embedding. Because we reuse this same `tower` model
for both inputs, the weights are shared automatically.

In [ ]:
tower = Sequential([
    Input(shape=(IMG_H, IMG_W, 1)),
    Conv2D(32, (3, 3), activation='relu', padding='same'), BatchNormalization(), MaxPooling2D(),
    Conv2D(64, (3, 3), activation='relu', padding='same'), BatchNormalization(), MaxPooling2D(),
    Conv2D(128, (3, 3), activation='relu', padding='same'), BatchNormalization(), MaxPooling2D(),
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(128),
], name='tower')
tower.summary()

## 6. Assemble the Siamese model

Two inputs go through the same tower, then a Lambda layer computes the Euclidean distance between
the two embeddings.

In [ ]:
def euclidean_distance(vecs):
    a, b = vecs
    return tf.sqrt(tf.reduce_sum(tf.square(a - b), axis=1, keepdims=True) + 1e-9)

input_a = Input(shape=(IMG_H, IMG_W, 1))
input_b = Input(shape=(IMG_H, IMG_W, 1))

emb_a = tower(input_a)
emb_b = tower(input_b)
distance = Lambda(euclidean_distance)([emb_a, emb_b])

model = Model([input_a, input_b], distance)

## 7. Contrastive loss & training

Genuine pairs (y=0) are pulled together; forged pairs (y=1) are pushed apart up to a margin.

In [ ]:
def contrastive_loss(y_true, d):
    margin = 1.0
    y_true = tf.cast(y_true, tf.float32)
    return tf.reduce_mean((1 - y_true) * tf.square(d) +
                          y_true * tf.square(tf.maximum(margin - d, 0)))

model.compile(optimizer=Adam(1e-3), loss=contrastive_loss)

early_stop = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)

history = model.fit([X1_tr, X2_tr], y_tr,
                    validation_data=([X1_va, X2_va], y_va),
                    epochs=30, batch_size=32,
                    callbacks=[early_stop, reduce_lr])

In [ ]:
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title('Contrastive loss'); plt.legend(); plt.show()

## 8. Pick a threshold

The model gives a distance, not a yes/no. We pick the cutoff on the validation set at the Equal
Error Rate (where false accepts = false rejects). Distance below the cutoff -> genuine.

In [ ]:
val_d = model.predict([X1_va, X2_va]).ravel()

# EER: find the threshold where false-accept rate ~ false-reject rate
fpr, tpr, thr = roc_curve(y_va, val_d)
fnr = 1 - tpr
eer_idx = np.nanargmin(np.abs(fpr - fnr))
threshold = thr[eer_idx]
print('threshold:', round(float(threshold), 4))
print('val EER:  ', round(float((fpr[eer_idx] + fnr[eer_idx]) / 2) * 100, 2), '%')

## 9. Evaluate on unseen writers

In [ ]:
test_d = model.predict([X1_te, X2_te]).ravel()
pred = (test_d > threshold).astype(int)   # far -> forged

acc = (pred == y_te).mean()
auc = roc_auc_score(y_te, test_d)
far = (test_d[y_te == 1] < threshold).mean()   # forgery accepted as genuine
frr = (test_d[y_te == 0] > threshold).mean()   # genuine rejected as forged

print('Accuracy:', round(acc * 100, 2), '%')
print('ROC-AUC :', round(auc, 3))
print('FAR     :', round(far * 100, 2), '%')
print('FRR     :', round(frr * 100, 2), '%')

In [ ]:
# how well do the two distance distributions separate?
plt.hist(test_d[y_te == 0], bins=40, alpha=0.6, label='genuine')
plt.hist(test_d[y_te == 1], bins=40, alpha=0.6, label='forged')
plt.axvline(threshold, color='k', ls='--', label='threshold')
plt.xlabel('distance'); plt.legend(); plt.show()

## 10. Save the tower + threshold

We save the tower (not the full pair-model) so the Streamlit app can embed one signature at a time.

In [ ]:
tower.save('siamese_embedding.keras')
json.dump({'threshold': float(threshold), 'img_h': IMG_H, 'img_w': IMG_W, 'model': 'siamese_cnn'},
          open('siamese_cnn_meta.json', 'w'))
print('saved tower + meta')

## 11. Takeaways

- The Siamese network learns an actual distance metric, so genuine and forged pairs land in
  different parts of the embedding space — and it holds up on writers it never trained on.
- We can enroll a brand new person from a single reference signature, no retraining needed.
- The tower is still trained from scratch on fairly little data.
- **Next:** swap the from-scratch tower for a pretrained MobileNetV2 (transfer learning) and see
  if borrowed ImageNet features push the numbers up.